# Post-class assignment: MLIP for Li-ion diffusion

**MS.49900 AI for Materials Science, Hands-on session 7 (post-class)**

| Part | Question |
|---|---|
| B | How fast does Li diffuse in Li₆PS₅Cl, and what is its activation energy? |
| C | How many DFT structures does fine-tuning need? |
| D *(optional)* | Can a committee of models flag its own errors? |

Run on a T4 GPU. Part B takes about 15 min (three 10 ps MD runs), Part C about 4 min, Part D about 3 min.
Write your answers in the *Your answer* cells and submit this notebook with the figures.

## Setup

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.')

In [ ]:
# Install SevenNet and download the course repository with the Li6PS5Cl DFT data (about 1 min)
!pip install -q sevenn==0.13.0

import os
if not os.path.exists('MS49900-AI4M'):
    !git clone -q https://github.com/kwongibaek/MS49900-AI4M.git

DATA = 'MS49900-AI4M/Data'
print(sorted(f for f in os.listdir(DATA) if f.startswith(('lpsc_', 'sevennet_'))))

In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ase.io import read
from ase.units import bar
from sevenn.calculator import SevenNetCalculator

to_kbar = 1 / bar / 1000          # eV/Å³ -> kbar


def dft_labels(traj):
    """DFT energy (eV/atom), forces (eV/Å) and stress (kbar) stored in the data file."""
    E = np.array([a.get_potential_energy() / len(a) for a in traj])
    F = np.concatenate([a.get_forces().ravel() for a in traj])
    S = np.concatenate([a.get_stress().ravel() for a in traj]) * to_kbar
    return E, F, S


def predict(calc, traj):
    """MLIP energy, forces and stress for every structure, plus seconds per structure."""
    E, F, S = [], [], []
    warm = traj[0].copy(); warm.calc = calc; warm.get_forces()   # warm-up, not timed
    t0 = time.time()
    for atoms in traj:
        a = atoms.copy()          # copy keeps the DFT labels of the original
        a.calc = calc
        E.append(a.get_potential_energy() / len(a))
        F.append(a.get_forces().ravel())
        S.append(a.get_stress().ravel() * to_kbar)
    sec = (time.time() - t0) / len(traj)
    return np.array(E), np.concatenate(F), np.concatenate(S), sec


def free_gpu(calc=None):
    del calc
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

import sevenn.util as util
from copy import deepcopy
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.optimize import LBFGS
from ase.filters import FrechetCellFilter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part B. Li diffusion and activation energy

Run NVT MD at three temperatures, get the Li diffusion coefficient from the MSD (Einstein relation), and fit an Arrhenius law.

$$\mathrm{MSD}(t) = 6Dt, \qquad D(T) = D_0 \exp\left(-\frac{E_a}{k_B T}\right)$$

**Model.** Use your fine-tuned checkpoint from class (upload `checkpoint_fine_tuned.pth`), or the provided one.

In [ ]:
import os
FT_PATH = 'checkpoint_fine_tuned.pth'
if not os.path.exists(FT_PATH):
    FT_PATH = f'{DATA}/lpsc_finetuned_checkpoint.pth'
    print('Using the provided fine-tuned checkpoint')
print(FT_PATH)

# Settings
TEMPS = [600, 900, 1200]       # K
MD_STEPS = 5000                # 2 fs x 5000 = 10 ps per temperature
FIT_START = 2.0                # ps, skip the start of each run

In [ ]:
def relax(atoms, calc, fmax=0.05, steps=1000):
    a = atoms.copy(); a.calc = calc
    opt = LBFGS(FrechetCellFilter(a, hydrostatic_strain=True), logfile=None)
    opt.run(fmax=fmax, steps=steps)
    return a

def run_nvt(atoms, calc, T, n_steps, dt_fs=2.0, interval=10, seed=0):
    a = atoms.copy(); a.calc = calc
    MaxwellBoltzmannDistribution(a, temperature_K=T, rng=np.random.default_rng(seed))
    Stationary(a)
    dyn = Langevin(a, dt_fs * units.fs, temperature_K=T, friction=0.01 / units.fs,
                   rng=np.random.default_rng(seed + 1))
    pos, temp = [], []                 # the observer also runs at step 0
    def record():
        pos.append(a.get_positions().copy()); temp.append(a.get_temperature())
    dyn.attach(record, interval=interval)
    dyn.run(n_steps)
    t = np.arange(len(pos)) * interval * dt_fs / 1000
    return t, np.array(pos), np.array(temp)

def msd_of(pos, atoms, element):
    m = atoms.get_masses()
    disp = pos - pos[0]
    disp -= (disp * m[None, :, None]).sum(1, keepdims=True) / m.sum()   # remove drift
    sym = np.array(atoms.get_chemical_symbols())
    return (disp[:, sym == element] ** 2).sum(-1).mean(1)

calc = SevenNetCalculator(FT_PATH)
start = relax(read(f'{DATA}/lpsc_test_md.extxyz.gz', 0), calc)
print('relaxed volume per unit cell:', round(start.get_volume() / 8, 1), 'Å³')

runs, paths = {}, {}
for T in TEMPS:
    t0 = time.time()
    t, pos, temp = run_nvt(start, calc, T, MD_STEPS)
    runs[T] = (t, msd_of(pos, start, 'Li'), msd_of(pos, start, 'S'))
    paths[T] = pos[::2]                                  # keep positions for the Li-path plot
    print(f'{T} K done in {time.time() - t0:.0f} s, mean T = {temp[len(temp)//5:].mean():.0f} K')
free_gpu(calc)

**Blank 1.** Get D from the slope of the Li MSD.

In [ ]:
kB = 8.617333e-5   # eV/K

def diffusion_coefficient(t, msd_li, t_start=FIT_START):
    fit = t >= t_start
# You can change here------------------------------------------------------------
    slope, intercept = ...   # linear fit of MSD vs time in the fit window (Å²/ps)
    D = ...                  # Einstein relation in 3D, then Å²/ps -> cm²/s
# -------------------------------------------------------------------------------
    return D, slope, intercept

D_list = {}
fig, ax = plt.subplots(figsize=(6, 4))
for T, (t, msd_li, msd_s) in runs.items():
    D, slope, intercept = diffusion_coefficient(t, msd_li)
    D_list[T] = D
    line, = ax.plot(t, msd_li, label=f'Li {T} K')
    ax.plot(t, msd_s, ls=':', color=line.get_color())
    ax.plot(t[t >= FIT_START], slope * t[t >= FIT_START] + intercept, 'k--', lw=0.8)
ax.set_xlabel('Time (ps)'); ax.set_ylabel('MSD (Å²)'); ax.set_title('Li (solid), S (dotted)')
ax.legend(); plt.tight_layout(); plt.show()

pd.Series(D_list, name='D_Li (cm²/s)').to_frame()

### Where do the Li ions go?

Top view of the bottom half of the cell: S, P and Cl at the start, and the path of every Li ion during the run.

In [ ]:
from ase.data import atomic_numbers
from ase.data.colors import jmol_colors

ATOM_COLORS = {el: jmol_colors[atomic_numbers[el]] for el in ['Li', 'P', 'S', 'Cl']}

def plot_li_paths(ax, atoms, pos, title):
    sym = np.array(atoms.get_chemical_symbols())
    sel = pos[0, :, 2] < atoms.cell[2, 2] / 2
    for el in ['S', 'P', 'Cl']:
        m = (sym == el) & sel
        ax.scatter(pos[0, m, 0], pos[0, m, 1], s=40, color=ATOM_COLORS[el], edgecolors='k', linewidths=0.3, label=el)
    for i in np.where((sym == 'Li') & sel)[0]:
        ax.plot(pos[:, i, 0], pos[:, i, 1], color='#A05EE0', lw=0.6)
    ax.plot([], [], color='#A05EE0', lw=2, label='Li path')
    ax.set_aspect('equal'); ax.set_xlabel('x (Å)'); ax.set_ylabel('y (Å)'); ax.set_title(title)

fig, axes = plt.subplots(1, len(paths), figsize=(5.5 * len(paths), 5.5), squeeze=False)
for ax, (T, p) in zip(axes[0], paths.items()):
    plot_li_paths(ax, start, p, f'{T} K, {MD_STEPS * 2 / 1000:g} ps')
axes[0, -1].legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.savefig('partB_li_paths.png', dpi=150); plt.show()

**Blank 2.** Fit the Arrhenius law and get the activation energy.

In [ ]:
T_arr = np.array(list(D_list), dtype=float)
D_arr = np.array(list(D_list.values()))
# You can change here------------------------------------------------------------
slope, intercept = ...   # fit ln(D) vs 1/T
Ea = ...                 # activation energy (eV)
# -------------------------------------------------------------------------------

x = 1000 / T_arr
plt.figure(figsize=(5, 4))
plt.semilogy(x, D_arr, 'o', color='#2F5597')
xx = np.linspace(x.min() * 0.95, x.max() * 1.05, 50)
plt.semilogy(xx, np.exp(intercept + slope * xx / 1000), 'k--')
plt.xlabel('1000 / T (1/K)'); plt.ylabel('D_Li (cm²/s)')
plt.title(f'Arrhenius fit: Ea = {Ea:.3f} eV')
plt.tight_layout(); plt.savefig('partB_arrhenius.png', dpi=150); plt.show()
print(f'Ea = {Ea:.3f} eV')

**Questions B**

1. Compare your $E_a$ with a reported value for Li₆PS₅Cl (cite the paper). Is it higher or lower? Use the Li-path plots: how do the paths change from 600 K to 1200 K?
2. Name two reasons your value could be off (think about simulation length, cell size, temperature range and the model).
3. *(Optional)* Set the model to `'7net-0'` and repeat. Does a uMLIP give a different $E_a$? Why can it still run stable MD even with its 2.4 eV/atom energy offset?

*Your answer:*

## Part C. How much data does fine-tuning need?

Fine-tune `7net-0-small` with 8, 17, 42 and 80 structures (same validation set, same 10 epochs) and plot the test error against the number of training structures.

In [ ]:
from torch_geometric.loader import DataLoader
from sevenn.train.graph_dataset import SevenNetGraphDataset
from sevenn.train.trainer import Trainer
from sevenn.error_recorder import ErrorRecorder

# Settings
N_TRAIN_LIST = [8, 17, 42, 80]
EPOCHS = 10
N_VALID = 10
N_TEST = 20

base_ckpt = f'{DATA}/sevennet_0_small_checkpoint.pth'
_, base_config = util.model_from_checkpoint(base_ckpt)
dataset = SevenNetGraphDataset(
    cutoff=base_config['cutoff'], root='sevenn_data', drop_info=False,
    files=[f'{DATA}/lpsc_600K.extxyz.gz', f'{DATA}/lpsc_1200K.extxyz.gz'],
)
perm = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(0))
valid_set = dataset[perm[:N_VALID]]
pool = perm[N_VALID:]

test_all = read(f'{DATA}/lpsc_test_md.extxyz.gz', ':')
test = test_all[::len(test_all) // N_TEST][:N_TEST]
dft_E, dft_F, dft_S = dft_labels(test)


def finetune(n_train, out_path, epochs=EPOCHS, indices=None):
    """Fine-tune 7net-0-small on the first n_train structures of the pool (or on given indices)."""
    torch.manual_seed(0)
    model, config = util.model_from_checkpoint(base_ckpt)
    config.update({
        'device': device, 'is_ddp': False,
        'optimizer': 'adam', 'optim_param': {'lr': 0.004},
        'scheduler': 'linearlr',
        'scheduler_param': {'start_factor': 1.0, 'total_iters': epochs, 'end_factor': 0.0001},
        'loss': 'huber', 'loss_param': {'delta': 0.01},
        'force_loss_weight': 0.1, 'stress_loss_weight': 0.01,
    })
    trainer = Trainer.from_config(model, config)
    rec = ErrorRecorder.from_config(config)
    train_idx = pool[:n_train] if indices is None else indices
    train_loader = DataLoader(dataset[train_idx], batch_size=4, shuffle=True)
    for _ in range(epochs):
        trainer.run_one_epoch(train_loader, is_train=True, error_recorder=rec)
        trainer.scheduler_step()
        rec.epoch_forward()
    trainer.write_checkpoint(out_path, config=config, epoch=epochs)
    free_gpu()
    return out_path

**Blank 3.** Fine-tune and evaluate for each training-set size.

In [ ]:
curve = {}
calc = SevenNetCalculator(base_ckpt)                 # 0 structures = pretrained only
E, F, S, _ = predict(calc, test); free_gpu(calc)
curve[0] = {'Energy': rmse(dft_E, E), 'Force': rmse(dft_F, F), 'Stress': rmse(dft_S, S)}

for n in N_TRAIN_LIST:
    t0 = time.time()
# You can change here------------------------------------------------------------
    # fine-tune with n structures, then predict the test set with the new checkpoint
    E, F, S = ...
# -------------------------------------------------------------------------------
    curve[n] = {'Energy': rmse(dft_E, E), 'Force': rmse(dft_F, F), 'Stress': rmse(dft_S, S)}
    print(f'{n:3d} structures done in {time.time() - t0:.0f} s')

curve = pd.DataFrame(curve).T
curve.index.name = 'Training structures'
curve.round(4)

In [ ]:
units_ = {'Energy': 'eV/atom', 'Force': 'eV/Å', 'Stress': 'kbar'}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
n = curve.index.values[1:]
for ax, col in zip(axes, curve.columns):
    ax.semilogx(n, curve[col].values[1:], 'o-', color='#2F5597', label='fine-tuned')
    ax.axhline(curve[col].iloc[0], color='k', ls='--', lw=1, label='pretrained')
    ax.set_xticks(n); ax.set_xticklabels([str(v) for v in n]); ax.minorticks_off()
    ax.set_ylim(bottom=0)
    ax.set_xlabel('Training structures'); ax.set_ylabel(f'{col} RMSE ({units_[col]})')
    ax.set_title(col)
axes[0].legend()
plt.tight_layout(); plt.savefig('partC_learning_curve.png', dpi=150); plt.show()

**Questions C**

1. How many structures are enough for each property? Which property needs the least data?
2. With 8 structures the model gets far fewer optimizer steps than with 80 (same epochs). How would you make the comparison fairer?
3. One DFT MD snapshot of this cell costs hours of CPU time. Based on your curve, how many would you compute for a new electrolyte?

*Your answer:*

## Part D (optional). Can the models tell when they are unsure?

Fine-tune three models on different random sets of 42 structures (a **committee**). Where the three disagree, the prediction is less trustworthy. Check whether the committee spread follows the true force error.

This is the idea behind **active learning**: compute new DFT data only where the committee disagrees. About 3 min on a T4.

In [ ]:
N_COMMITTEE = 3
committee = []
for k in range(N_COMMITTEE):
    g = torch.Generator().manual_seed(100 + k)
    subset = pool[torch.randperm(len(pool), generator=g)[:42]]        # a different random set for each model
    committee.append(finetune(42, f'committee_{k}.pth', indices=subset))
    print(f'model {k} done')

Fs = []
for path in committee:
    calc = SevenNetCalculator(path)
    Fs.append(predict(calc, test)[1].reshape(-1, 3))
    free_gpu(calc)
Fs = np.array(Fs)                                          # models x atoms x 3
spread = Fs.std(axis=0).mean(axis=1)                       # committee disagreement per atom (eV/Å)
error = np.linalg.norm(Fs.mean(axis=0) - dft_F.reshape(-1, 3), axis=1)   # true force error per atom (eV/Å)
symbols = np.concatenate([a.get_chemical_symbols() for a in test])
r = np.corrcoef(spread, error)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
hb = axes[0].hexbin(spread, error, gridsize=40, bins='log', cmap='Blues', mincnt=1)
axes[0].set_xlabel('Committee spread (eV/Å)'); axes[0].set_ylabel('True force error (eV/Å)')
axes[0].set_title(f'Per atom, correlation r = {r:.2f}'); fig.colorbar(hb, ax=axes[0], label='atoms')
edges = np.quantile(spread, np.linspace(0, 1, 11))
idx = np.clip(np.digitize(spread, edges[1:-1]), 0, 9)
axes[1].plot([spread[idx == i].mean() for i in range(10)], [error[idx == i].mean() for i in range(10)], 'o-', color='#2F5597')
axes[1].set_xlabel('Committee spread, binned (eV/Å)'); axes[1].set_ylabel('Mean true error (eV/Å)')
axes[1].set_title('Larger spread, larger error?')
plt.tight_layout(); plt.savefig('partD_committee.png', dpi=150); plt.show()

pd.DataFrame({'spread': spread, 'error': error, 'element': symbols}).groupby('element').mean().round(3)

**Questions D**

1. Is the committee spread a useful warning sign for large errors? Use the correlation and the binned plot.
2. Which element has the largest spread and error? Why might that be?
3. Describe how you would use the committee to choose the next 10 structures to compute with DFT.

*Your answer:*

## Submission

Submit this notebook (with outputs) and the three figures: `partB_li_paths.png`, `partB_arrhenius.png`, `partC_learning_curve.png` (and `partD_committee.png` if you did Part D).

In [ ]:
try:
    from google.colab import files
    for f in ['partB_li_paths.png', 'partB_arrhenius.png', 'partC_learning_curve.png', 'partD_committee.png']:
        if os.path.exists(f):
            files.download(f)
except ImportError:
    pass